# Symptom extraction workbench

Run the (citation-validated) extraction on **real** transcripts and examine how well it
pulls symptom facts — what it **captures, misses, or misattributes**. This is the main
task-focused notebook. Symptoms now; other clinical-note facts later.

Everything runs here in the notebook. The extraction uses the real model
(`openai/gpt-5.6-luna` via OpenRouter), so **each run in section 2 costs one API call**.

In [ ]:
from clinical import (
    get_transcript,
    load_transcripts,
    show_symptom_evidence_matrix,
    show_transcript_coverage,
)
from graphs.validation_loop import build_graph

ENCOUNTER_DATE = "27.11.2025"
EXTRACTOR = build_graph()  # the validation-loop graph, real model

print(f"{len(load_transcripts())} transcripts available (transcript_1 … transcript_105)")

## 1. Pick a transcript and read it

Change `TID` to explore different consultations.

In [ ]:
TID = "transcript_1"
transcript = get_transcript(TID)
print(transcript)

## 2. Run the extraction  ⚠️ calls the model (one API call)

In [ ]:
result = EXTRACTOR.invoke({"transcript": transcript, "encounter_date": ENCOUNTER_DATE})
extraction = result["extraction"]

print(f"attempts={result['attempts']}  unresolved citation errors={len(result['errors'])}")
print(f"problems extracted: {len(extraction.problems)}")
for p in extraction.problems:
    print(f"  - {p.problem}  [{p.status}]"
          f"{'  laterality=' + p.laterality if p.laterality else ''}")

## 3. Examine the structured facts

Per problem: each characteristic (quality, severity, onset, …) with its supporting
transcript evidence.

In [ ]:
show_symptom_evidence_matrix(extraction)

## 4. Examine coverage — what did it actually cite?

The transcript with every cited excerpt **highlighted**. The *un-highlighted* text is what
the extractor did **not** pick up — read those spans to spot missed symptoms or
characteristics. Anything listed as "not found verbatim" is a citation-quality problem.

In [ ]:
show_transcript_coverage(transcript, extraction)

## 5. Critique — what's missed or wrong?

Read the transcript against the extracted facts (sections 3 & 4) and note issues. Failure
modes to check for:

- **Missed symptoms** — a complaint discussed but not captured as a problem or associated symptom.
- **Misattributed evidence** — a characteristic attached to the wrong problem.
- **Laterality / anatomy** — `vasak` / `parem` / `bilateraalne` wrong or missing.
- **Negation & uncertainty** — a denied symptom recorded as present (or vice versa).
- **Temporality** — a proposed/hypothetical event treated as completed.
- **Over- / under-splitting** — one problem split into many, or several merged into one.

**Notes for this transcript:**
- …

## Iterate

- Tune the schema (`clinical/schema.py`) or prompt (`clinical/prompts.py`), then re-run
  sections 2 → 4 to see the effect.
- Change `TID` to sample other consultations.
- For the full model trace (exact prompts, the retry), open LangSmith → project `agent_01`.